# Volatility Dynamics 10 — physical jump model vs 0DTE option wings

**VD 019. Cross-branch reuse approved by Ivan (2026-10-10):** VRP branch Black-76 conventions (`option_pricing`) and its pricing-audit findings (VRP 006). Deep-OTM 0DTE quotes are tick-priced, so the comparison is in **price space against bid/ask**, never exchange IV. No session after the VRP confirmation boundary (2026-11-06) is used.

Each two-sided OTM 0DTE quote is priced under a compound-Poisson + diffusion model:

$$P^{jump}=\sum_{n=0}^{3}\text{Pois}(n;\Lambda)\,\mathbb E\big[\text{B76}(F e^{S_n-\Lambda(\bar m-1)},K,D)\big]$$

- $D$ is the **ex-ante** diffusive variance to expiry: backward 4h bipower × Module 08 clock, with settlement-TWAP weights.
- $\Lambda$ is the expected number of jumps.
- The jumps are measured signed jump sizes (local-σ units) × current σ.
- Jump rate and sizes are **leave-one-session-out**.

**Pre-registered (VD 019 §4):**
- **Sample** — non-floor two-sided OTM quotes, TTE 60–1200 min, 15-min grid, assets not flagged by Module 09 P2.
- **Primary** — moneyness $z=|\ln K/F|/\sqrt D\in[2,5)$; $M$ = median over sessions of the session-median $\ln(\text{mid}/P^{jump})$.
- **Verdict** — **RICH** if the session-bootstrap 95% CI > 0, **CHEAP** if < 0, else **CONSISTENT**.
- **Also reported** — all buckets, per asset, no-jump model, sensitivities (diffusion ×0.85, λ ×0.5/×2), and hold-to-settlement P&L. Outputs: `10/`.

In [1]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_jump_hawkes as jh
from data_engineering.modeling import volatility_jump_compound as jc
from data_engineering.modeling import volatility_jump_option_wings as ow
START = os.getenv('VD10_START', '').strip() or None
END = os.getenv('VD10_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD10_ASSETS', '').split(',') if x.strip()]
N_BOOT = int(os.getenv('VD10_N_BOOT', str(ow.N_BOOT)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '10'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 10', ow.VERSION, ow.selftest())


Module 10 0.1.0 {'fast_b76_matches_option_pricing': True, 'put_call_parity': True, 'zero_jump_is_black76': True, 'monte_carlo_agrees': True, 'jumps_add_wing_value': True, 'settlement_weights': True, 'session_statistic': True}


In [2]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

df = load_frame(['futures_index_price'], START, END, ASSETS)
df = df[df.state_time < ow.VRP_CONFIRMATION_START].reset_index(drop=True)
grid = vb.build_grid(df)
jres = jc.run(grid)
excluded = jres['ticks'].loc[jres['ticks'].tick_artifact, 'underlying'].tolist()
print('Module 09 P2 tick-artifact exclusions:', excluded)
panel, S, Z, J, valid = jres['panel'], jres['S'], jres['Z'], jres['J'], jres['valid']
BVb = ow.backward_bipower(panel, S)
sess_bar = ow.session_of_bar(panel)
used = [a for a in panel.assets if a not in excluded]
params = ow.loso_jump_params(panel, J, valid, Z, panel.R, sess_bar, used)
print('sessions with LOSO parameters:', len(params))


rows 42,336 | assets ['AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'TRX', 'XRP'] | 2026-09-19 08:00:00+00:00 -> 2026-10-10 07:55:00+00:00
Module 09 P2 tick-artifact exclusions: []
sessions with LOSO parameters: 22


In [3]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
need = ['state_id', 'state_time', 'underlying', 'option_family_id', 'option_type', 'strike', 'best_bid_price',
        'best_ask_price', 'option_forward_price', 'minutes_to_expiry', 'instrument_type']
miss = sorted(set(need) - cols)
if miss:
    raise RuntimeError(f'normalized_option_chain lacks {miss}')
chain = con.execute(f"""
    WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
    SELECT {', '.join('n.' + c for c in need)}
    FROM normalized_option_chain n
    JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
    WHERE n.minutes_to_expiry BETWEEN {ow.MIN_TTE_MIN} AND {ow.MAX_TTE_MIN}
      AND EXTRACT(minute FROM n.state_time) % {ow.QUOTE_GRID_MIN} = 0
      AND n.state_time < TIMESTAMPTZ '{ow.VRP_CONFIRMATION_START.isoformat()}'
      AND ((lower(substr(n.option_type, 1, 1)) = 'c' AND n.strike > n.option_forward_price)
        OR (lower(substr(n.option_type, 1, 1)) = 'p' AND n.strike < n.option_forward_price))
""").df()
con.close()
chain = chain[chain.underlying.isin(used)]
quotes = ow.prepare_quotes(chain)
floors = ow.tick_floors(quotes)
display(floors)
t0 = time.time()
q = ow.attach_model(quotes, panel, S, BVb, params)
q = ow.flag_floor(q, floors)
print(f'priced {len(q):,} quotes in {time.time()-t0:.0f}s | floor-bound share {q.floor_bound.mean():.3f}')


,underlying,instrument_type,min_tick_native,share_bid_at_tick
0,AVAX,linear,0.00050,0.076173
1,BTC,linear,5.00000,0.142551
2,ETH,linear,0.20000,0.117002
3,HYPE,linear,0.00200,0.086804
4,SOL,linear,0.10000,0.304005
5,TRX,linear,0.00005,0.139601
6,XRP,linear,0.00050,0.154612


priced 76,405 quotes in 11s | floor-bound share 0.132


In [4]:
prim = ow.primary_sample(q, excluded)
primary = ow.session_median_stat(prim, 'log_mid_over_jump', n_boot=N_BOOT)
nojump = ow.session_median_stat(prim, 'log_mid_over_diff', n_boot=N_BOOT)
print('PRIMARY (z in [2,5)):', json.dumps(primary, indent=2, default=float))
print('same rows vs diffusion-only model:', json.dumps(nojump, indent=2, default=float))
sens = []
for label, ds, ls in [('diffusion_x0.85', 0.85, 1.0), ('lambda_x0.5', 1.0, 0.5), ('lambda_x2', 1.0, 2.0)]:
    x = prim.copy()
    x['lr'] = ow.reprice(prim, params, diff_scale=ds, lam_scale=ls)
    sens.append({'sensitivity': label, **ow.session_median_stat(x, 'lr', n_boot=N_BOOT)})
sens = pd.DataFrame(sens)
display(sens.round(4))
by_asset = pd.DataFrame([{'underlying': u, **ow.session_median_stat(g, 'log_mid_over_jump', n_boot=1000)}
                         for u, g in prim.groupby('underlying')])
display(by_asset.round(4))


PRIMARY (z in [2,5)): {
  "n_rows": 14805,
  "n_sessions": 19,
  "M": 2.956626311328909,
  "ci_low": 2.4965829736135627,
  "ci_high": 3.1712665630603323,
  "verdict": "RICH",
  "mean_ratio_mid_over_model": 2108.9817222605425
}
same rows vs diffusion-only model: {
  "n_rows": 14805,
  "n_sessions": 19,
  "M": 3.7330008141273754,
  "ci_low": 3.1331002250628597,
  "ci_high": 3.9647508236585223,
  "verdict": "RICH",
  "mean_ratio_mid_over_model": 21985.699655869546
}


,sensitivity,n_rows,n_sessions,M,ci_low,ci_high,verdict,mean_ratio_mid_over_model
0,diffusion_x0.85,14805,19,3.5317,2.9955,3.7806,RICH,9217.7570
1,lambda_x0.5,14805,19,3.2961,2.7859,3.4915,RICH,4293.8511
2,lambda_x2,14805,19,2.5639,2.0881,2.7886,RICH,1119.3243


,underlying,n_rows,n_sessions,M,ci_low,ci_high,verdict,mean_ratio_mid_over_model
0,AVAX,3878,19,2.0815,1.1695,2.2732,RICH,602.9142
1,BTC,2704,19,2.6725,2.5202,3.3776,RICH,1552.7240
2,ETH,3286,19,3.0102,2.6636,4.1523,RICH,3156.2886
3,HYPE,2451,19,2.4600,1.9714,3.0180,RICH,3013.4020
4,SOL,79,10,2.7114,2.2072,3.0247,RICH,34.3301
5,TRX,1658,19,4.2755,3.0898,4.9304,RICH,3715.9073
6,XRP,749,19,2.6515,2.2169,3.1445,RICH,1022.3073


In [5]:
buckets = ow.bucket_table(q, n_boot=1000)
display(buckets.round(4))
piv = buckets.pivot(index='z_bucket', columns='tte_bucket', values='jump_M')
display(piv.round(3))
settle = ow.settlement_prices(grid, sorted(q.session.unique()))
pnl = ow.settlement_pnl(q, settle)
display(pnl.round(3))


,z_bucket,tte_bucket,jump_n_rows,jump_n_sessions,jump_M,jump_ci_low,jump_ci_high,jump_verdict,jump_mean_ratio_mid_over_model,diff_M,share_model_below_bid,share_model_above_ask,share_floor_bound
0,"[0.0, 1.0)","(120.0, 240.0]",1461,19,0.1113,-0.0535,0.3208,CONSISTENT,1.120000e+00,0.2140,0.2635,0.3265,0.0171
1,"[0.0, 1.0)","(240.0, 480.0]",4563,19,0.0250,-0.1342,0.2824,CONSISTENT,1.118700e+00,0.1457,0.3471,0.3458,0.0171
2,"[0.0, 1.0)","(480.0, 960.0]",14101,19,-0.0284,-0.2466,0.1796,CONSISTENT,1.040200e+00,0.0727,0.3271,0.4829,0.0171
3,"[0.0, 1.0)","(60.0, 120.0]",392,18,0.3252,0.1251,0.4261,RICH,1.354200e+00,0.4133,0.2449,0.1760,0.0171
4,"[0.0, 1.0)","(960.0, 1200.0]",8938,19,0.0914,0.0193,0.1257,RICH,1.113600e+00,0.2106,0.4128,0.4090,0.0171
5,"[1.0, 2.0)","(120.0, 240.0]",706,19,0.7779,0.4494,1.0808,RICH,2.776900e+00,1.0914,0.5283,0.1034,0.1090
6,"[1.0, 2.0)","(240.0, 480.0]",2806,19,0.8131,0.4119,1.0086,RICH,2.669500e+00,1.1537,0.6044,0.1016,0.1090
7,"[1.0, 2.0)","(480.0, 960.0]",10227,19,0.4209,0.2508,0.8021,RICH,2.353400e+00,0.6745,0.5430,0.2209,0.1090
8,"[1.0, 2.0)","(60.0, 120.0]",169,17,0.8576,0.6502,1.5074,RICH,3.179000e+00,1.2830,0.4852,0.0414,0.1090
9,"[1.0, 2.0)","(960.0, 1200.0]",6959,19,0.5938,0.4670,0.7065,RICH,2.494200e+00,0.8856,0.6195,0.1805,0.1090


tte_bucket,"(120.0, 240.0]","(240.0, 480.0]","(480.0, 960.0]","(60.0, 120.0]","(960.0, 1200.0]"
z_bucket,,,,,
"[0.0, 1.0)",0.111,0.025,-0.028,0.325,0.091
"[1.0, 2.0)",0.778,0.813,0.421,0.858,0.594
"[2.0, 3.0)",2.440,2.233,2.129,2.364,2.193
"[3.0, 5.0)",5.738,5.657,5.275,5.512,5.549
"[5.0, 8.0)",12.826,13.940,12.937,NaN,14.823
"[8.0, inf)",62.403,31.649,34.350,22.352,33.366


,z_bucket,metric,n_rows,n_sessions,mean_of_session_means,ci_low,ci_high
0,"[0.0, 1.0)",pnl_buy_ask_bps,29511,19,-14.647,-25.173,-0.898
1,"[0.0, 1.0)",pnl_sell_bid_bps,29511,19,3.619,-9.461,14.432
2,"[0.0, 1.0)",model_edge_buy_bps,29511,19,-0.286,-5.678,5.791
3,"[1.0, 2.0)",pnl_buy_ask_bps,20891,19,-10.688,-16.272,-3.686
4,"[1.0, 2.0)",pnl_sell_bid_bps,20891,19,3.006,-4.259,8.493
5,"[1.0, 2.0)",model_edge_buy_bps,20891,19,-8.890,-11.083,-6.582
6,"[2.0, 3.0)",pnl_buy_ask_bps,9673,19,-8.583,-10.350,-6.491
7,"[2.0, 3.0)",pnl_sell_bid_bps,9673,19,2.871,1.064,4.246
8,"[2.0, 3.0)",model_edge_buy_bps,9673,19,-9.318,-10.461,-8.200
9,"[3.0, 5.0)",pnl_buy_ask_bps,5132,19,-6.798,-8.568,-5.130


In [6]:
q.drop(columns=[c for c in ['instrument_name'] if c in q]).to_csv(OUT / '10_priced_quotes.csv.gz', index=False, compression='gzip')
floors.to_csv(OUT / '10_tick_floors.csv', index=False)
buckets.to_csv(OUT / '10_bucket_table.csv', index=False)
by_asset.to_csv(OUT / '10_primary_by_asset.csv', index=False)
sens.to_csv(OUT / '10_primary_sensitivities.csv', index=False)
pnl.to_csv(OUT / '10_settlement_pnl.csv', index=False)
pd.DataFrame([{'session': s, 'n_jumps_used': p['n_jumps_used'], **{f'lambda_{a}': v for a, v in p['lambda_bar'].items()}}
              for s, p in params.items()]).to_csv(OUT / '10_loso_jump_params.csv', index=False)
decision = {'primary_z_2_5': primary, 'same_rows_diffusion_only': nojump, 'excluded_assets_tick_artifact': excluded}
(OUT / '10_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {'version': ow.VERSION, 'module': '10_jump_model_vs_option_wings',
        'cross_branch_reuse': 'approved by Ivan 2026-10-10: VRP option_pricing conventions + VRP 006 audit findings',
        'vrp_confirmation_start_excluded': str(ow.VRP_CONFIRMATION_START), 'tte_min': [ow.MIN_TTE_MIN, ow.MAX_TTE_MIN],
        'quote_grid_min': ow.QUOTE_GRID_MIN, 'primary_z': list(ow.PRIMARY_Z), 'n_quotes_priced': int(len(q)),
        'n_primary_rows': int(len(prim)), 'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max())}
(OUT / '10_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(decision, indent=2, default=float))


{
  "primary_z_2_5": {
    "n_rows": 14805,
    "n_sessions": 19,
    "M": 2.956626311328909,
    "ci_low": 2.4965829736135627,
    "ci_high": 3.1712665630603323,
    "verdict": "RICH",
    "mean_ratio_mid_over_model": 2108.9817222605425
  },
  "same_rows_diffusion_only": {
    "n_rows": 14805,
    "n_sessions": 19,
    "M": 3.7330008141273754,
    "ci_low": 3.1331002250628597,
    "ci_high": 3.9647508236585223,
    "verdict": "RICH",
    "mean_ratio_mid_over_model": 21985.699655869546
  },
  "excluded_assets_tick_artifact": []
}
